# Documentation and resources

**Big Query**
- Colab has an example notebook on BigQuery too.  File > Open notebook > Examples > Getting Started with BigQuery.
- Also a Big Query Snippets Example Notebook
- [BigQuery Documentation]( https://cloud.google.com/bigquery/docs )
- [Open Data Sets]( https://console.cloud.google.com/marketplace/browse?filter=solution-type:dataset )
- [Reddit - list of data sets available on BQ]( https://www.reddit.com/r/bigquery/wiki/datasets )

**Big Query Console**  
- [Google Cloud Console]( https://console.cloud.google.com )
- Make sure your project is selected
- Scroll down to BigQuery on the left menu
- [Setup and query instructions]( https://cloud.google.com/bigquery/docs/quickstarts/query-public-dataset-console )

**SQL**
- [Kaggle Intro to SQL]( https://www.kaggle.com/learn/intro-to-sql ) uses BigQuery
- [Kaggle Advanced SQL]( https://www.kaggle.com/learn/advanced-sql )

**BigQuery Public Data Sets**
- [Austin Crime]( https://console.cloud.google.com/marketplace/product/city-of-austin/austin-crime )


#  Linking BigQuery to Colab

## Getting started

**You will only need to do this part once.**

1. Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.

    - Create Project
    - Project Name

2. [Enable BigQuery APIs](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) for the project

**Note:** You get 1 TB/month of free queries for open datasets
- Kaggle gives you 5 TB/month free



## Imports

In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata


In [2]:
%alias gcloud gcloud

### Provide your credentials

In [3]:
auth.authenticate_user()
print('Authenticated')

Authenticated


## Optional: Enable data table display

Colab includes the ``google.colab.data_table`` package that can be used to display large pandas dataframes as an interactive data table.
It can be enabled with:

In [4]:
%load_ext google.colab.data_table
# %unload_ext google.colab.data_table

If you would prefer to return to the classic Pandas dataframe display, you can disable this by running:

In [ ]:
%unload_ext google.colab.data_table

## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [5]:
gcloud projects list --sort-by=projectId

PROJECT_ID              NAME           PROJECT_NUMBER  ENVIRONMENT
modular-tube-514623-f0  my-ds-project  342411619811


To take a quick anonymous survey, run:
  $ gcloud survey



## Declare the Cloud project ID which will be used throughout this notebook

Create a secret in your vault named `bq_billing_project_id` and assign it the value of the project ID you want to use.

1. open the vault ( key icon on the far left )
1. click "Add new secret"
1. type the name ( `bq_billing_project_id` )
1. paste the value ( do NOT type enter )
1. click back in the notebook


Fetch the project ID from your vault.

In [6]:
project_id = userdata.get('bq_billing_project_id')
project_id

'modular-tube-514623-f0'

## Samples data set



The [GSOD table](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=samples&t=gsod&page=table) in the Samples data set contains weather information collected by NOAA, such as precipitation amounts and wind speeds from late 1929 to early 2010.


# Use BigQuery via magics



The `google.cloud.bigquery` library also includes a magic command which runs a query and either displays the result or saves it to a variable as a `DataFrame`.

In [7]:
# Display query output immediately

%%bigquery --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

,total_rows
0,114420316


In [8]:
# Save output in a variable `df`

%%bigquery bq_public_data_df --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

In [9]:
bq_public_data_df

,total_rows
0,114420316


In [10]:
f'{bq_public_data_df.iloc[0,0]:,.0f}'

'114,420,316'

# Use BigQuery through google-cloud-bigquery



See [BigQuery documentation](https://cloud.google.com/bigquery/docs) and [library reference documentation](https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/usage.html).


### Count total number of rows

In [11]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  bq_df = client.query( query ).to_dataframe()


In [12]:
bq_df.shape


(1, 1)

In [13]:
bq_df

,total
0,114420316


In [14]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  row_count = client.query( query ).to_dataframe()["total"][0]

print(f'Full dataset has {row_count:,.0f} rows')


Full dataset has 114,420,316 rows


## Sample approximately 2000 random rows

### Describe the sampled data

In [15]:
sample_count = 2000

query = f'''
  SELECT
    *
  FROM
    `bigquery-public-data.samples.gsod`
  WHERE RAND() < {sample_count}/{row_count}
'''

with bigquery.Client(project=project_id) as client:
  df = client.query( query ).to_dataframe()

df


,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,337110,99999,1969,4,3,38.000000,8,29.400000,8,1024.599976,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
1,245070,99999,1975,7,4,50.400002,8,39.700001,8,1011.599976,...,NaN,<NA>,0.04,NaN,False,False,False,False,False,False
2,722615,22001,1978,3,31,69.000000,24,54.700001,24,1018.200012,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
3,948930,99999,1991,3,20,57.500000,5,47.099998,5,1015.799988,...,NaN,<NA>,0.01,NaN,False,False,False,False,False,False
4,324080,99999,1993,6,7,48.799999,9,36.799999,9,1010.799988,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2038,985430,99999,2008,2,15,77.800003,20,73.400002,20,1012.500000,...,NaN,<NA>,0.36,NaN,False,False,False,False,False,False
2039,711500,99999,2008,5,29,54.299999,24,NaN,<NA>,1016.799988,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
2040,75240,99999,2008,4,2,56.099998,24,49.200001,24,1030.300049,...,NaN,<NA>,0.02,NaN,False,False,False,False,False,False
2041,764580,99999,2009,9,21,86.199997,8,77.800003,8,1010.400024,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False


In [16]:
df.index.size

2043

In [17]:
df.describe().transpose().astype({"count": int})

,count,mean,std,min,25%,50%,75%,max
station_number,2043,520200.992658,298160.484127,10014.0,266725.0,560790.0,725361.5,999999.0
wban_number,2043,91014.933431,25302.226533,24.0,99999.0,99999.0,99999.0,99999.0
year,2043,1986.745472,16.403453,1937.0,1977.0,1989.0,2000.0,2010.0
month,2043,6.419481,3.479583,1.0,3.0,6.0,9.0,12.0
day,2043,15.43906,8.903805,1.0,8.0,15.0,23.0,31.0
mean_temp,2043,51.267597,24.855307,-89.599998,37.799999,54.599998,70.0,102.5
num_mean_temp_samples,2043,12.940773,7.918161,4.0,7.0,8.0,23.0,24.0
mean_dew_point,1947,40.815511,22.794645,-74.5,28.9,42.700001,56.099998,81.800003
num_mean_dew_point_samples,1947,12.820236,7.914272,4.0,7.0,8.0,23.0,24.0
mean_sealevel_pressure,1574,1014.691931,9.641985,951.200012,1009.400024,1014.299988,1020.274994,1053.699951


### View the first 10 rows

In [18]:
df.head(10)

,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,337110,99999,1969,4,3,38.000000,8,29.400000,8,1024.599976,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
1,245070,99999,1975,7,4,50.400002,8,39.700001,8,1011.599976,...,NaN,<NA>,0.04,NaN,False,False,False,False,False,False
2,722615,22001,1978,3,31,69.000000,24,54.700001,24,1018.200012,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
3,948930,99999,1991,3,20,57.500000,5,47.099998,5,1015.799988,...,NaN,<NA>,0.01,NaN,False,False,False,False,False,False
4,324080,99999,1993,6,7,48.799999,9,36.799999,9,1010.799988,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
5,20120,99999,1997,12,2,-3.100000,5,-5.900000,5,1024.900024,...,NaN,<NA>,0.00,4.3,False,False,False,False,False,False
6,226950,99999,1998,9,4,39.000000,7,35.200001,7,1018.299988,...,NaN,<NA>,0.01,NaN,False,False,False,False,False,False
7,703210,99999,2004,7,11,66.900002,24,52.599998,24,NaN,...,NaN,<NA>,0.00,NaN,True,True,True,True,True,True
8,725458,99999,2008,11,10,26.200001,24,22.600000,24,NaN,...,NaN,<NA>,0.00,NaN,False,False,False,False,False,False
9,63100,99999,2009,12,31,33.200001,24,30.299999,24,995.900024,...,NaN,<NA>,0.25,NaN,False,False,False,False,False,False


In [19]:
df.isnull().sum()

,0
station_number,0
wban_number,0
year,0
month,0
day,0
mean_temp,0
num_mean_temp_samples,0
mean_dew_point,96
num_mean_dew_point_samples,96
mean_sealevel_pressure,469


In [20]:
# 10 highest total_precipitation samples
(
df
  .sort_values('total_precipitation', ascending=False)
  .head(10)
  [['station_number', 'year', 'month', 'day', 'total_precipitation']]
)


,station_number,year,month,day,total_precipitation
1324,574610,2009,8,17,4.19
856,649310,1988,5,25,3.58
1643,476750,1953,10,1,3.54
1934,689060,1978,1,23,3.15
80,318010,1986,7,19,3.15
206,339760,1963,2,15,2.95
1993,722536,1988,6,28,2.72
1120,592710,1994,8,17,2.48
292,727925,2008,11,12,2.42
1913,672370,1988,2,2,2.36


# Use BigQuery through pandas-gbq



The `pandas-gbq` library is a community led project by the pandas community. It covers basic functionality, such as writing a DataFrame to BigQuery and running a query, but as a third-party library it may not handle all BigQuery features or use cases.

[Pandas GBQ Documentation]( https://pandas-gbq.readthedocs.io/en/latest/ )

In [21]:
query = '''
  SELECT
    name, SUM(number) as count
  FROM
    bigquery-public-data.usa_names.usa_1910_2013
  WHERE
    state = 'TX'
  GROUP BY
    name
  ORDER BY
    count DESC
  LIMIT
    100
'''

df = pandas_gbq.read_gbq( query, project_id=project_id, )
df.head()


Downloading: 100%|██████████|


,name,count
0,James,272793
1,John,235139
2,Michael,225320
3,Robert,220399
4,David,219028
